In [20]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [21]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0.2
)

In [22]:
class ChatState(TypedDict):
    message : Annotated[list[BaseMessage], add_messages]

In [23]:
def chat_func(state : ChatState):
    message = state["message"]
    response = llm.invoke(message)
    return {
        "message" : [response]
    }

In [24]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_bot", chat_func)

graph.add_edge(START, "chat_bot")
graph.add_edge("chat_bot", END)

chatbot = graph.compile(checkpointer = checkpointer)

In [ ]:
thread_id = 1
config = {"configurable" : {"thread_id" : thread_id}}
while True:
    user_message = input("user message")
    print("user : ", user_message)
    if user_message.strip().lower() in ["bye", "exit"]:
        break
    response = chatbot.invoke({"message" : [HumanMessage(content=user_message)]}, config=config)
    print("AI", response["message"][-1].content)


user :  hello buddy what is my name
AI Your name is Rahul.
user :  exit
